# ¿Mejora LightGBM con las probabilidades de FullNightNet?
Comparamos cuatro modelos sobre Sleep Cassette. La métrica principal es el F1 macro medio de los cinco pliegues históricos, con igual peso por pliegue. Medimos también F1 por etapa, exactitud y kappa. El segundo conjunto conserva los 16 pacientes de prueba utilizados en trabajos anteriores del proyecto. Ninguno intervino en estos ajustes.

La referencia es el LightGBM publicado con 435 variables y cuatro canales. La red directa de seis clases produce una distribución condicional sobre W, N1, N2, N3 y REM. Las dos variantes híbridas usan esas cinco probabilidades y una de ellas añade la probabilidad absoluta de movimiento. Una evaluación adicional respeta la decisión nativa de la red y cuenta una salida M como error cuando el experto anotó una etapa.

In [1]:
from pathlib import Path
import json
import pandas as pd

root = Path.cwd()
if root.name == 'notebook':
    root = root.parent
results = root / 'output/evidencia/resumen/fullnight-lgbm-w30'
metrics = pd.read_csv(results / 'metrics.csv')
intervals = pd.read_csv(results / 'paired_intervals.csv')
scores = pd.read_csv(results / 'leaderboard.csv').set_index('arm')
runs = json.loads((results / 'mlflow-runs.json').read_text())

## Separación de pacientes y reproducción
Los 62 pacientes de desarrollo aportan 122 noches. En cada pliegue externo usamos el LightGBM y la red neuronal históricos para puntuar a los pacientes de validación. Para las filas de entrenamiento de los híbridos se ajustaron diez redes internas en PyTorch, una por pareja de pliegues excluidos. Cada predicción interna proviene de una red que desconocía tanto al paciente predicho como a los pacientes del pliegue externo.

Las redes internas comparten el presupuesto original de dos pasadas por la señal. Se ejecutaron en MPS. Los árboles heredan en cada pliegue los parámetros, pesos de clase, preprocesamiento, semilla e iteraciones del LightGBM histórico. Para prueba, los híbridos se ajustan con las probabilidades externas de los cinco pliegues de desarrollo.

Los comandos reproducen el entrenamiento, la evaluación y el registro con las campañas de origen restauradas en `var/research` y el modelo publicado en `var/cloud/bundle`. En un equipo NVIDIA puede usarse `--device cuda`. El notebook permite consultar toda la comparación desde las tablas que acompañan la rama.

In [2]:
commands = [
    'scripts/stack_fullnight_folds.py --device auto',
    'scripts/compare_fullnight_lgbm.py',
    'scripts/compare_fullnight_edf.py',
    'scripts/report_fullnight_lgbm.py',
]
for cmd in commands:
    print('.venv/bin/python', cmd)

.venv/bin/python scripts/stack_fullnight_folds.py --device auto
.venv/bin/python scripts/compare_fullnight_lgbm.py
.venv/bin/python scripts/compare_fullnight_edf.py
.venv/bin/python scripts/report_fullnight_lgbm.py


## Ventanas de evaluación
W30 conserva las épocas desde 30 minutos antes del primer sueño anotado hasta 30 minutos después del último. Para replicar LightGBM, su contexto se calcula dentro de ese recorte e incluye las épocas de movimiento disponibles antes de filtrar las etiquetas. FullNightNet observa la señal nocturna completa en su forma habitual. En el análisis de EDF completo, el contexto tabular se calcula sobre todas las épocas elegibles antes de consultar al experto. El subconjunto `whole_edf_w30` permite observar el efecto del contexto conservando exactamente las mismas épocas puntuadas.

In [3]:
display(metrics.query("arm == 'lightgbm_release'")[['sample','subjects','nights','epochs']].set_index('sample'))
display(metrics[metrics['sample'].str.startswith('fold-')].pivot(index='sample', columns='arm', values='macro_f1').round(4))

,subjects,nights,epochs
sample,,,
fold-0,13,25,29788
fold-1,13,26,33150
fold-2,12,24,33483
fold-3,12,23,28353
fold-4,12,24,27278
development,62,122,152052
reserved,16,31,43417
whole_edf,16,31,83875
whole_edf_w30,16,31,43417


arm,fullnight_five,fullnight_native,lightgbm_release,stack_five,stack_five_movement
sample,,,,,
fold-0,0.6478,0.6462,0.7546,0.7621,0.7619
fold-1,0.5672,0.5673,0.7976,0.8031,0.8042
fold-2,0.6258,0.6261,0.7992,0.8031,0.8018
fold-3,0.6210,0.6216,0.8042,0.8076,0.8064
fold-4,0.6677,0.6673,0.7906,0.7943,0.7963


## Comparaciones pareadas
Los intervalos del 95 % se calculan con 2.000 remuestreos de participantes completos. Las dos noches de cada participante permanecen juntas. La diferencia se calcula sobre las predicciones agrupadas de los modelos ajustados, con el mismo denominador para ambos.

In [4]:
display(intervals.query("sample == 'reserved'")[['candidate','reference','metric','delta','ci95_low','ci95_high']].round(4))
display(metrics.query("sample == 'reserved'").set_index('arm')[['f1_W','f1_N1','f1_N2','f1_N3','f1_REM','movement_calls']].round(4))

,candidate,reference,metric,delta,ci95_low,ci95_high
8,fullnight_five,lightgbm_release,macro_f1,-0.1038,-0.1410,-0.0664
9,fullnight_five,lightgbm_release,f1_N1,-0.1088,-0.1465,-0.0725
10,stack_five,lightgbm_release,macro_f1,0.0092,0.0035,0.0154
11,stack_five,lightgbm_release,f1_N1,0.0170,0.0061,0.0282
12,stack_five_movement,lightgbm_release,macro_f1,0.0083,0.0028,0.0142
13,stack_five_movement,lightgbm_release,f1_N1,0.0123,0.0016,0.0219
14,stack_five_movement,stack_five,macro_f1,-0.0009,-0.0033,0.0019
15,stack_five_movement,stack_five,f1_N1,-0.0048,-0.0138,0.0042


,f1_W,f1_N1,f1_N2,f1_N3,f1_REM,movement_calls
arm,,,,,,
lightgbm_release,0.9238,0.5156,0.8489,0.7935,0.8575,0
fullnight_five,0.8935,0.4068,0.7515,0.7121,0.6564,0
stack_five,0.9314,0.5326,0.8523,0.8090,0.8597,0
stack_five_movement,0.9312,0.5278,0.8526,0.8079,0.8612,0
fullnight_native,0.8921,0.4049,0.7515,0.7138,0.6591,233


## Resultados y seguimiento
Las columnas de desarrollo presentan media y desviación estándar entre los cinco pliegues con el mismo divisor que la campaña histórica (cinco). Las columnas de prueba agrupan las épocas de las 31 noches reservadas. Cada almacén de MLflow conserva una corrida padre y las cinco evaluaciones, con métricas, predicciones por época, intervalos, modelos y auditoría de sujetos.

In [5]:
display(scores.round(4))
winner = scores.loc[['lightgbm_release','fullnight_five','stack_five','stack_five_movement']].cv_macro_f1.idxmax()
print(f'Mayor F1 macro medio en desarrollo: {winner} ({scores.loc[winner, "cv_macro_f1"]:.4f} ± {scores.loc[winner, "cv_sd"]:.4f}).')
for store, entry in runs.items():
    print(store, entry['experiment_name'], entry['parent'])

,cv_macro_f1,cv_sd,test_macro_f1,test_accuracy,test_f1_N1,test_kappa,edf_macro_f1,edf_f1_N1
arm,,,,,,,,
stack_five_movement,0.7941,0.0165,0.7961,0.8412,0.5278,0.7784,0.7790,0.4773
stack_five,0.7940,0.0165,0.7970,0.8410,0.5326,0.7784,0.7806,0.4841
lightgbm_release,0.7893,0.0179,0.7879,0.8338,0.5156,0.7684,0.7630,0.4530
fullnight_five,0.6259,0.0337,0.6840,0.7457,0.4068,0.6537,0.6981,0.4033
fullnight_native,0.6257,0.0334,0.6843,0.7438,0.4049,0.6516,0.6985,0.4015


Mayor F1 macro medio en desarrollo: stack_five_movement (0.7941 ± 0.0165).
lightgbm sleep-cassette-lgbm-bilstm-refinement-v1 bb7fa78abcfb41ab87c86ee89755ae87
fullnight ibio-fullnight 5ef260a86a0349e48a9ec6a956ec5348


La comparación distingue el valor de las probabilidades neuronales del aporte adicional de movimiento. Para interpretar la generalización se deben leer las diferencias pareadas en prueba junto con el promedio de desarrollo. La evaluación se limita a Sleep Cassette con los cuatro canales compatibles con la versión publicada.

El híbrido con movimiento obtuvo el mayor promedio en desarrollo (0,7941). El híbrido con las cinco probabilidades de etapa obtuvo el mayor F1 en prueba W30 (0,7970) y EDF completo (0,7806). En ambos protocolos, el intervalo pareado frente a LightGBM quedó por encima de cero. La diferencia incremental al añadir movimiento fue compatible con cero. El siguiente contraste para el artículo sería en una cohorte externa con los cuatro canales de la versión publicada.